In [1]:
import sys, pickle
from pathlib import Path
import pandas as pd

# 1. Repo Discovery (Boilerplate to find the root 'syllable_generation' folder)
# ---------------------------------------------------------------------
# Assuming notebook is in: .../syllable_generation/projects/jonathan_openfield_example/
current_dir = Path.cwd() 
repo_root = current_dir.parents[1] # Go up 2 levels to find repo root
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

# 2. Load the "Dirty" Sessions
# ---------------------------------------------------------------------
# Adjust this path if you saved the pickle somewhere else!
# Assuming you saved it in the standard 'loaders' folder:
DIRTY_PICKLE_PATH = repo_root / "data" / "sessions_dirty.pkl"

if not DIRTY_PICKLE_PATH.exists():
    raise FileNotFoundError(f"Could not find your dirty pickle at: {DIRTY_PICKLE_PATH}")

print(f"Loading dirty sessions from: {DIRTY_PICKLE_PATH}")
with open(DIRTY_PICKLE_PATH, "rb") as f:
    sessions = pickle.load(f)

print(f"Loaded {len(sessions)} sessions.")

# 3. Define Output Path for exclusions.csv
# ---------------------------------------------------------------------
# Since we are in 'projects/jonathan_openfield_example', overrides is right here.
OUTPUT_CSV = current_dir / "overrides" / "exclusions.csv"

# Ensure the folder exists
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
print(f"Exclusions will be saved to: {OUTPUT_CSV}")

Loading dirty sessions from: /home/fsp585/Desktop/FiberphotometryCode/monoamine_interactions/data/sessions_dirty.pkl
Loaded 12 sessions.
Exclusions will be saved to: /home/fsp585/Desktop/FiberphotometryCode/monoamine_interactions/projects/jonathan_oft_baseline/overrides/exclusions.csv


In [2]:
import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# --- Aux Function ---
def map_sessions_to_npz(sessions, npz_data):
    mapping = {}
    npz_keys = list(npz_data.files)
    for s in sessions:
        match = next((k for k in npz_keys if s.session_id in k), None)
        if match: mapping[s.session_id] = match
    return mapping

# --- Main Class ---
class PhotometryCleaner:
    def __init__(self, sessions, output_path, npz_path=None):
        self.sessions = sorted(sessions, key=lambda s: s.session_id)
        self.output_path = Path(output_path).resolve()
        self.npz_path = Path(npz_path).resolve() if npz_path else None
        self.symlink_path = self.output_path.parent / "auto_mask.npz"
        self.reviewed_path = self.output_path.parent / "reviewed_sessions.txt"
        
        self.exclusions = {} 
        self.npz_data = None
        self.session_to_npz_key = {}
        self.reviewed = set()
        
        # Load Data
        self._load_existing_csv()
        self._apply_pre_exclusions()
        self._load_reviewed()
        self._init_npz()
        
        # State
        self.current_sess_idx = 0
        self.current_signals = {} 
        self.current_sig_key = None 
        
        # --- UI Elements ---
        self.out = widgets.Output()
        
        # Initialize Dropdown with Indicators
        self.drop_session = widgets.Dropdown(
            options=self._get_session_options(), 
            value=0, 
            description='Session:', 
            layout=widgets.Layout(width='400px') # Slightly wider for the checkmark
        )
        self.drop_signal = widgets.Dropdown(description='Signal:', layout=widgets.Layout(width='200px'))
        
        self.chk_reviewed = widgets.Checkbox(value=False, description='Mark Reviewed', indent=False, layout=widgets.Layout(width='120px'))
        self.prog_label = widgets.Label(value=f"({len(self.reviewed)}/{len(self.sessions)})")

        self.btn_prev_sess = widgets.Button(description="< Sess", layout=widgets.Layout(width='70px'))
        self.btn_next_sess = widgets.Button(description="Sess >", layout=widgets.Layout(width='70px'))
        self.btn_next_sig = widgets.Button(description="Next Signal >", icon='arrow-right')

        self.btn_mode_pan = widgets.Button(description="Pan/Zoom", icon='arrows', button_style='')
        self.btn_mode_select = widgets.Button(description="Box Select", icon='crosshairs', button_style='info')

        self.btn_save = widgets.Button(description="Save Config", button_style='success', icon='save')
        self.btn_undo = widgets.Button(description="Undo Last", button_style='warning', icon='undo')
        
        mode_text = "<b>Auto-Mask: ON</b> (Grey traces visible)" if self.npz_data else "<b>Auto-Mask: OFF</b> (Manual mode)"
        self.info_label = widgets.HTML(value=f"<i>{mode_text}</i>")

        # --- Bindings ---
        self.drop_session.observe(self._on_sess_change, names='value')
        self.drop_signal.observe(self._on_sig_change, names='value')
        self.chk_reviewed.observe(self._on_reviewed_change, names='value')
        
        self.btn_prev_sess.on_click(lambda b: self._nav_sess(-1))
        self.btn_next_sess.on_click(lambda b: self._nav_sess(1))
        self.btn_next_sig.on_click(self._nav_sig)
        
        self.btn_mode_pan.on_click(self._set_mode_pan)
        self.btn_mode_select.on_click(self._set_mode_select)
        
        self.btn_save.on_click(self._save_config)
        self.btn_undo.on_click(self._undo_last)

        # --- Layout ---
        self.layout = widgets.VBox([
            widgets.HBox([self.btn_prev_sess, self.drop_session, self.btn_next_sess, self.chk_reviewed, self.prog_label]),
            widgets.HBox([self.drop_signal, self.btn_next_sig, widgets.Label(" | Tools: "), self.btn_mode_pan, self.btn_mode_select]),
            self.out,
            widgets.HBox([self.btn_save, self.btn_undo, self.info_label])
        ])
        
        self._load_session_data(0)

    def _get_session_options(self):
        """Generates dropdown list with checkmarks for reviewed sessions."""
        opts = []
        for i, s in enumerate(self.sessions):
            # Add checkmark if reviewed
            prefix = "✔️ " if s.session_id in self.reviewed else ""
            label = f"{prefix}{s.session_id}"
            opts.append((label, i))
        return opts

    def _init_npz(self):
        if not self.npz_path or not self.npz_path.exists(): return
        try:
            self.npz_data = np.load(self.npz_path)
            self.session_to_npz_key = map_sessions_to_npz(self.sessions, self.npz_data)
        except Exception as e: print(f"Error loading NPZ: {e}")

    def _load_existing_csv(self):
        if self.output_path.exists():
            try:
                df = pd.read_csv(self.output_path, dtype=str).fillna("")
                for _, row in df.iterrows():
                    aid, reg, intervals = row.get("ID"), row.get("Region"), row.get("Intervals", "")
                    if aid and reg and intervals:
                        if aid not in self.exclusions: self.exclusions[aid] = {}
                        if reg not in self.exclusions[aid]: self.exclusions[aid][reg] = []
                        for part in intervals.split(";"):
                            if "idx:" in part:
                                try:
                                    rng = part.replace("idx:", "").strip().split("-")
                                    self.exclusions[aid][reg].append((float(rng[0]), float(rng[1])))
                                except: pass
            except Exception: pass

    def _load_reviewed(self):
        if self.reviewed_path.exists():
            try:
                with open(self.reviewed_path, 'r') as f:
                    self.reviewed = set(line.strip() for line in f if line.strip())
            except: pass

    def _get_signals_data(self, session):
        signals = {}
        sig_meta = session.metadata.get('signal_meta', {})
        phot = getattr(session, 'photometry', {})
        
        valid_set = None
        if self.npz_data and session.session_id in self.session_to_npz_key:
            npz_key = self.session_to_npz_key[session.session_id]
            valid_set = set(self.npz_data[npz_key])
            
        for sig_col, meta in sig_meta.items():
            orig_letter = meta.get('orig_letter', 'G').upper()
            target_key = '470' if orig_letter == 'G' else ('560' if orig_letter == 'R' else '415')
            
            if phot and target_key in phot:
                df = phot[target_key]
                reg = meta.get('brain_reg', 'Unknown')
                hemi = meta.get('hemisphere', '')
                lbl = f"{reg}-{hemi}" if hemi else reg
                unique = f"{sig_col}_{orig_letter}"
                
                raw_y = df[sig_col].values
                frames = df['frame_idx'].values
                
                masked_y = raw_y.copy()
                if valid_set is not None:
                    mask = np.isin(frames, list(valid_set))
                    masked_y[~mask] = np.nan
                
                try:
                    import sys; parent_dir = str(self.output_path.parent.parent); sys.path.append(parent_dir) if parent_dir not in sys.path else None
                    import config as pconfig
                    vr = getattr(pconfig, 'VALID_FRAME_RANGE', None)
                    if vr:
                        mask_range = (frames >= vr[0]) & (frames <= vr[1])
                        valid_y = masked_y[mask_range]
                        if len(valid_y) > 0:
                            y_min, y_max = np.nanmin(valid_y), np.nanmax(valid_y)
                        else:
                            y_min, y_max = np.nanmin(masked_y), np.nanmax(masked_y)
                    else:
                        y_min, y_max = np.nanmin(masked_y), np.nanmax(masked_y)
                except:
                    y_min, y_max = np.nanmin(masked_y), np.nanmax(masked_y)
                if np.isnan(y_min): y_min, y_max = np.nanmin(raw_y), np.nanmax(raw_y)
                if np.isnan(y_min): y_min, y_max = 0, 1
                
                yrange = y_max - y_min
                y_min -= yrange * 0.05
                y_max += yrange * 0.05
                
                signals[unique] = {
                    'x': frames, 
                    'y_raw': raw_y,     
                    'y_masked': masked_y, 
                    'ylims': [y_min, y_max],
                    'reg_key': lbl, 
                    'display': f"{lbl} ({orig_letter})",
                    'color': '#2ca02c' if orig_letter == 'G' else '#d62728'
                }
        return signals

    def _load_session_data(self, idx):
        self.current_sess_idx = idx
        session = self.sessions[idx]
        self.current_signals = self._get_signals_data(session)
        
        # Sync Reviewed UI without triggering events
        self.chk_reviewed.unobserve(self._on_reviewed_change, names='value')
        self.chk_reviewed.value = session.session_id in self.reviewed
        self.chk_reviewed.observe(self._on_reviewed_change, names='value')
        
        opts = [(v['display'], k) for k, v in self.current_signals.items()]
        
        self.drop_signal.unobserve(self._on_sig_change, names='value')
        self.drop_signal.options = opts
        if opts:
            self.drop_signal.value = opts[0][1]
            self.current_sig_key = opts[0][1]
        else:
            self.drop_signal.value = None
            self.current_sig_key = None
        self.drop_signal.observe(self._on_sig_change, names='value')
        
        self.render_plot()

    def render_plot(self):
        self.out.clear_output(wait=True)
        if not self.current_signals or not self.current_sig_key:
            with self.out: print("No Data Available")
            return

        session = self.sessions[self.current_sess_idx]
        aid = session.metadata.get('animal_id', 'Unknown')
        data = self.current_signals[self.current_sig_key]
        
        mode = 'select' if self.btn_mode_select.button_style == 'info' else 'pan'
        
        fig = go.FigureWidget()
        fig.update_layout(
            title=f"<b>{aid}</b> | {data['display']}", 
            dragmode=mode, 
            height=500,
            width=1200, 
            xaxis=dict(fixedrange=False, range=[50000, 250000]),
            yaxis=dict(fixedrange=False, range=data['ylims']),
            template="plotly_white",
            margin=dict(l=10, r=10, t=40, b=10),
            showlegend=True
        )
        
        if self.npz_data:
            fig.add_trace(go.Scattergl(
                x=data['x'], y=data['y_raw'], 
                line=dict(color='lightgrey', width=1), opacity=0.5,
                name="Auto-Excluded", mode='lines', hoverinfo='skip'
            ))
        
        fig.add_trace(go.Scattergl(
            x=data['x'], y=data['y_masked'], 
            line=dict(color=data['color'], width=1.5), 
            name="Valid Signal", mode='lines'
        ))

        if aid in self.exclusions and data['reg_key'] in self.exclusions[aid]:
            for s, e in self.exclusions[aid][data['reg_key']]:
                fig.add_shape(type="rect", x0=s, x1=e, y0=0, y1=1, xref="x", yref="paper", fillcolor="black", opacity=0.2, line_width=0)

        def on_select(trace, points, selector):
            if not selector or not selector.xrange: return
            s, e = selector.xrange
            if aid not in self.exclusions: self.exclusions[aid] = {}
            if data['reg_key'] not in self.exclusions[aid]: self.exclusions[aid][data['reg_key']] = []
            self.exclusions[aid][data['reg_key']].append((round(s), round(e)))
            with fig.batch_update():
                fig.add_shape(type="rect", x0=s, x1=e, y0=0, y1=1, xref="x", yref="paper", fillcolor="red", opacity=0.4, line_width=0)
            self.info_label.value = f"Excluded: {int(s)}-{int(e)}"

        fig.data[-1].on_selection(on_select)
        self.current_fig = fig
        with self.out: display(fig)

    # --- Handlers ---
    def _nav_sess(self, d):
        new = self.current_sess_idx + d
        if 0 <= new < len(self.sessions): self.drop_session.value = new 
    def _on_sess_change(self, c): 
        if c['new'] != self.current_sess_idx: self._load_session_data(c['new'])
    def _on_sig_change(self, c): 
        self.current_sig_key = c['new']; self.render_plot()
    def _nav_sig(self, b):
        opts = [o[1] for o in self.drop_signal.options]
        try: self.drop_signal.value = opts[(opts.index(self.drop_signal.value) + 1) % len(opts)]
        except: pass
    def _set_mode_pan(self, b):
        self.btn_mode_pan.button_style, self.btn_mode_select.button_style = 'info', ''
        if hasattr(self, 'current_fig'): self.current_fig.update_layout(dragmode='pan')
    def _set_mode_select(self, b):
        self.btn_mode_select.button_style, self.btn_mode_pan.button_style = 'info', ''
        if hasattr(self, 'current_fig'): self.current_fig.update_layout(dragmode='select')
    def _undo_last(self, b):
        aid = self.sessions[self.current_sess_idx].metadata.get('animal_id')
        if not self.current_sig_key: return
        reg = self.current_signals[self.current_sig_key]['reg_key']
        if aid in self.exclusions and reg in self.exclusions[aid] and self.exclusions[aid][reg]:
            self.exclusions[aid][reg].pop(); self.render_plot()

    def _on_reviewed_change(self, change):
        sid = self.sessions[self.current_sess_idx].session_id
        if change['new']:
            self.reviewed.add(sid)
        else:
            self.reviewed.discard(sid)
        
        # Update Progress Label
        self.prog_label.value = f"({len(self.reviewed)}/{len(self.sessions)})"
        
        # Update Dropdown List (Dynamic)
        # Need to preserve current selection
        current_idx = self.drop_session.value
        self.drop_session.unobserve(self._on_sess_change, names='value')
        self.drop_session.options = self._get_session_options()
        self.drop_session.value = current_idx
        self.drop_session.observe(self._on_sess_change, names='value')

    def _save_config(self, b):
        # 1. Save CSV
        rows = []
        for aid, regs in self.exclusions.items():
            for r, intervals in regs.items():
                if intervals:
                    s_ints = "; ".join([f"idx:{int(s)}-{int(e)}" for s, e in sorted(intervals)])
                    rows.append({"ID": aid, "Region": r, "Intervals": s_ints})
        pd.DataFrame(rows).to_csv(self.output_path, index=False)
        
        # 2. Save Reviewed List
        with open(self.reviewed_path, "w") as f:
            for sid in sorted(self.reviewed):
                f.write(f"{sid}\n")

        # 3. Manage Symlink
        msg_suffix = ""
        if self.npz_path:
            if self.symlink_path.exists() or self.symlink_path.is_symlink():
                self.symlink_path.unlink()
            self.symlink_path.symlink_to(self.npz_path)
            msg_suffix = " + Auto-Mask Linked"
        else:
            if self.symlink_path.exists() or self.symlink_path.is_symlink():
                self.symlink_path.unlink()
                msg_suffix = " + Auto-Mask Removed"
        
        self.info_label.value = f"<b>Saved!</b> ({len(rows)} manual regions, {len(self.reviewed)} reviewed{msg_suffix})"

    def show(self): 
        display(self.layout)
    def _apply_pre_exclusions(self):
        try:
            import sys
            parent_dir = str(self.output_path.parent.parent)
            if parent_dir not in sys.path:
                sys.path.append(parent_dir)
            import config as pconfig
            valid_range = getattr(pconfig, 'VALID_FRAME_RANGE', None)
            if not valid_range: return
        except Exception as e:
            print('Could not load VALID_FRAME_RANGE:', e)
            return

        start_valid, end_valid = valid_range
        for session in self.sessions:
            sid = session.session_id
            if sid not in self.exclusions: self.exclusions[sid] = {}
            sig_meta = session.metadata.get('signal_meta', {})
            df = getattr(session, 'photometry', {}).get('combined_filtered')
            if df is None: continue
            max_frame = int(df['frame_idx'].max())

            for sig_col, meta in sig_meta.items():
                reg = meta.get('region', meta.get('brain_reg', 'Unknown'))
                hemi = meta.get('hemisphere', '')
                lbl = f"{reg}-{hemi}" if hemi else reg

                if lbl not in self.exclusions[sid]: 
                    self.exclusions[sid][lbl] = []

                # Convert to ints to avoid tuple comparison issues
                has_start = any((int(s) <= 0 and int(e) >= start_valid - 1) for s, e in self.exclusions[sid][lbl])
                if not has_start and start_valid > 0:
                    self.exclusions[sid][lbl].append((0.0, float(start_valid)))

                has_end = any((int(s) <= end_valid + 1 and int(e) >= max_frame) for s, e in self.exclusions[sid][lbl])
                if not has_end and max_frame > end_valid:
                    self.exclusions[sid][lbl].append((float(end_valid), float(max_frame)))


In [3]:
# 1. Define Paths
NPZ_PATH = Path('/home/fsp585/Desktop/FiberphotometryCode/syllable_generation/syllabizers/keypoint_moseq_syll/demo_project/full-run/video_frame_indexes.npz')
CSV_PATH = Path("./overrides/exclusions.csv")
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# 2. Launch
app = PhotometryCleaner(sessions, CSV_PATH, NPZ_PATH)
app.show()

In [ ]:
npz_data = np.load(NPZ_PATH)

In [5]:
for k in npz_data.keys():
    print(k)

Arena1234_CenterTopCamVid_2024-12-16T12_42_27_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T14_05_53_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T15_14_26_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T11_27_19_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-16T12_42_27_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-13T14_50_39_bottom_rightDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-13T13_37_45_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-13T13_37_45_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020
Arena1234_CenterTopCamVid_2024-12-1

In [6]:
sessions[0].session_id

'2024-12-13T12_20_06_bottom_left'